In [1]:
import os
os.chdir(os.environ['PWD'])

import numpy as np
import yaml
from utilities.utils import correct_type_of_entry, get_exp_file_name, create_all_configs, get_min_max_loss
import json
import pandas as pd
from itertools import product
import os
import matplotlib.pyplot as plt
import scienceplots
plt.style.use(['high-contrast'])

/Users/mathieubazinet/Documents/PythonProjects/ncp2l/ncp2lEnv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Quantization experiments

In [ ]:
dataset = "amazon"

dataset_config_name = "./configs/dataset_configs/" + dataset + ".yaml"
with open(dataset_config_name) as file:
    dataset_configuration = yaml.safe_load(file)

sweep_config_name = "./configs/experiment_configs/quantize_" + dataset + ".yaml"
with open(sweep_config_name) as file:
    sweep_configuration = yaml.safe_load(file)

hps = {}
for key, item in sweep_configuration['parameters'].items():
    if item.get('values', None) is not None:
        hps[key] = correct_type_of_entry(item['values'])
size_hyperparams = tuple([len(l) for l in hps.values()])

hp_list = list(hps.values())[1:]
params_product = list(product(*hp_list))
name_list = []
idx_list = []
for params in params_product:
    name = ""
    for p in params:
        name += str(p) + " "
    name_list.append(name[:-1])
    idx = ()
    for p_idx in range(len(params)):
        p_key = list(hps.keys())[1:][p_idx]
        idx += (hps[p_key].index(params[p_idx]),)
    idx_list.append(tuple(idx))

In [ ]:
use_delta = False
zero_one_loss = True

values_to_fetch = ['complement_error', 'validation_error', 'test_error']

if not use_delta:
    values_to_fetch += ['best_model_complement_error','best_model_test_error']
    
values_to_fetch += ['disagreement', 'disagreement_kl']

results_matrix = np.ones(((len(values_to_fetch),) + size_hyperparams))
results_matrix[:] = np.nan

for sweep_config in create_all_configs(sweep_configuration):
    file_name = get_exp_file_name(sweep_config|dataset_configuration, path="./amazon_pactl_logs/")
    if os.path.exists(file_name):
        with open(file_name) as f:
            config = json.load(f)
            for val_to_fetch_idx in range(len(values_to_fetch)):
                matrix_idx = tuple([val_to_fetch_idx] + [hps[key].index(config['config'].get(key, None)) for key in hps.keys()])
                val_to_fetch = values_to_fetch[val_to_fetch_idx]
                results_matrix[matrix_idx] = config.get(val_to_fetch, None)
                if use_delta:
                    if val_to_fetch == 'complement_error':
                        results_matrix[matrix_idx] -= config.get('best_model_complement_error', None)
                    elif val_to_fetch == 'test_error':
                        results_matrix[matrix_idx] -= config.get('best_model_test_error', None)

reshaped_matrix = np.nanmean(results_matrix, axis=1).reshape(results_matrix.shape[0],np.prod(results_matrix.shape[2:])).T
mean_df = pd.DataFrame(reshaped_matrix, index=name_list, columns=values_to_fetch)
mean_df = mean_df.dropna(axis='rows')

reshaped_matrix_std = np.nanstd(results_matrix, axis=1).reshape(results_matrix.shape[0],np.prod(results_matrix.shape[2:])).T
std_df = pd.DataFrame(reshaped_matrix_std, index=name_list, columns=values_to_fetch)
std_df = std_df.dropna(axis="rows")

mean_df

In [ ]:
rounding_val = 2 if zero_one_loss else 4
rounding = f"%.{rounding_val}f"

'complement_error', 'test_error', 'best_model_complement_error', 'best_model_test_error',

vals_to_add = [ 'QAT', 'nbit', 'complement_error', 'test_error']
if not use_delta:
    vals_to_add += ['best_model_complement_error', 'best_model_test_error']
vals_to_add += ['disagreement', 'disagreement_kl']

mult = 100.0 if zero_one_loss else 1.0

results_list = []
model_list = []

for i in range(mean_df.shape[0]):
    mean_vals = mean_df.iloc[i]
    std_vals = std_df.iloc[i]

    name = mean_vals.name
    first_index = name.index(" ")
    
    model_name = name[0:first_index]
    second_index = name[first_index+1:].index(" ")
    nbit = name[first_index:first_index+second_index+1]
    
    qat = name[first_index + second_index+1:]
    list_of_vals = []
    
    for val in vals_to_add:
        if val == "nbit":
            list_of_vals.append(nbit)
            continue
        elif val == "QAT":
            list_of_vals.append(qat)
            continue
        
        temp = (rounding % round(mult*mean_vals[val],rounding_val)) + r"$\pm$"
        temp += (rounding % round(mult*std_vals[val], rounding_val))
        list_of_vals.append(temp)
    
    results_list.append(pd.Series(list_of_vals, index=vals_to_add))
    model_list.append(model_name)

if zero_one_loss:
    df = pd.DataFrame(results_list, index=model_list)
    df = df.assign(idx=df.index).sort_values(by=["idx", "QAT"]).drop(columns="idx")
    print(df.to_latex(float_format="%.2f"))
else:
    print(pd.DataFrame(results_list, index=model_list).sort_values(by=["Model", "QAT"], inplace=True).to_latex(float_format="%.4f"))

In [ ]:
import seaborn as sn
import matplotlib.pyplot as plt

plt.figure(figsize=(4,2))


model = "DistilBERT"
get_message_len = False
_, _, files = next(os.walk("./amazon_pactl_logs_tamia/"))

seeds = [1, 2, 3, 4, 42]
nbits = [2, 4, 8]
qat = [True, False]

my_array = np.zeros((len(seeds), len(nbits), len(qat)))

for file in files:
    if model in file:
        with open("./amazon_pactl_logs/" + file) as f:
            config = json.load(f)
            index = (seeds.index(config['config']['seed']), nbits.index(config['config']['nbits']), qat.index(config['config']['qat']))
            if get_message_len:
                my_array[index] = config['message_len'] / 8e+6
            else:
                my_array[index] = config['disagreement_kl']

mean_array = np.mean(my_array, axis=0)
mean_array[0,0] = None

std_array = np.std(my_array, axis=0)

mean_std_array = np.empty(mean_array.shape, dtype=object)
for i in range(mean_array.shape[0]):
    for j in range(mean_array.shape[1]):
        if i == 0 and j == 0:
            mean_std_array[i,j] = "N/A"
            continue
        if get_message_len:
            mean_std_array[i,j] = f"{mean_array[i,j]:.2f}±{std_array[i,j]:.2f}"
        else:
            mean_std_array[i,j] = f"{100 * mean_array[i,j]:.2f}±{100 *std_array[i,j]:.2f}"

font_size = 13

if get_message_len:
    sn.heatmap(pd.DataFrame(mean_array, index=nbits, columns=qat),
               annot=pd.DataFrame(mean_std_array),
               fmt="", cmap="Blues")
else:
    sn.heatmap(pd.DataFrame(100 * mean_array, index=nbits, columns=qat),
               annot=pd.DataFrame(mean_std_array),
               fmt="", cmap="Blues",  vmin=0.0, vmax=43, #cbar=(model == "GPT2")
              )
arr = np.full(mean_array.shape, fill_value=None, dtype=float)
arr[0,0] = 0.0
annot = pd.DataFrame(arr, dtype=object)
annot.iloc[0,0] = "N/A"

sn.heatmap(pd.DataFrame(arr, index=nbits, columns=qat), annot=annot, fmt="", cbar=False)
sn.set(font_scale=1.2) 

plt.xlabel("Quantization-aware training")
plt.ylabel("Number of bits")

if get_message_len:
    plt.savefig(f"./results/heatmap_message_len_in_MB_{model}.pdf")
else:
    plt.savefig(f"./results/heatmap_disagreement_kl_{model}.pdf", bbox_inches='tight')

# PACTL experiments

In [ ]:
dataset = "amazon"

dataset_config_name = "./configs/dataset_configs/" + dataset + ".yaml"
with open(dataset_config_name) as file:
    dataset_configuration = yaml.safe_load(file)

sweep_config_name = "./configs/experiment_configs/pactl/" + dataset + ".yaml"
with open(sweep_config_name) as file:
    sweep_configuration = yaml.safe_load(file)

hps = {}
for key, item in sweep_configuration['parameters'].items():
    if item.get('values', None) is not None:
        hps[key] = correct_type_of_entry(item['values'])
size_hyperparams = tuple([len(l) for l in hps.values()])

hp_list = list(hps.values())[1:]
params_product = list(product(*hp_list))
name_list = []
idx_list = []
for params in params_product:
    name = ""
    for p in params:
        name += str(p) + " "
    name_list.append(name[:-1])
    idx = ()
    for p_idx in range(len(params)):
        p_key = list(hps.keys())[1:][p_idx]
        idx += (hps[p_key].index(params[p_idx]),)
    idx_list.append(tuple(idx))

def get_best_huber_loss(model_type):
    start_index = 1 if model_type == "DistilBERT" else 0
    df = pd.read_csv("./amazon_pactl_logs/huber_loss.csv").T
    df.columns = ["Train accuracy", "Test accuracy", "Train huber loss", "Test huber loss"]
    df = df.drop(index=["Unnamed: 0"] + [str(i) for i in range(start_index,10,2)])
    return df.mean(0), df.std(0)
    

In [ ]:
zero_one_loss = False

if zero_one_loss:
    values_to_fetch = ['complement_error', 'test_error', 'best_model_complement_error','best_model_test_error',\
                       'kl_bound', 'disagreement_kl', 'full_disagreement_bound_brute_force', 'message_len']
else:
    values_to_fetch = ['complement_huber_loss', 'test_huber_loss',
                       'huber_kl_bound', 'disagreement_softmax_huber', 'full_disagreement_softmax', 'message_len', "Train huber loss", "Test huber loss"]

results_matrix = np.ones(((len(values_to_fetch),) + size_hyperparams))
results_matrix[:] = np.nan

for sweep_config in create_all_configs(sweep_configuration):
    file_name = get_exp_file_name(sweep_config|dataset_configuration, path="./amazon_pactl_logs/")
    if not os.path.exists(file_name):
        print(file_name)
    if os.path.exists(file_name):
        with open(file_name) as f:
            config = json.load(f)
            for val_to_fetch_idx in range(len(values_to_fetch)):
                matrix_idx = tuple([val_to_fetch_idx] + [hps[key].index(config['config'].get(key, None)) for key in hps.keys()])
                val_to_fetch = values_to_fetch[val_to_fetch_idx]
                results_matrix[matrix_idx] = config.get(val_to_fetch, None)
                if val_to_fetch == "full_disagreement_bound_brute_force":
                    if config.get('full_disagreement_bound_brute_force') > config.get('kl_bound') + config.get('disagreement_kl'):
                        results_matrix[matrix_idx] = config.get('kl_bound') + config.get('disagreement_kl')

                if val_to_fetch == "message_len":
                    results_matrix[matrix_idx] /= 8000

reshaped_matrix = np.nanmean(results_matrix, axis=1).reshape(results_matrix.shape[0],np.prod(results_matrix.shape[2:])).T
mean_df = pd.DataFrame(reshaped_matrix, index=name_list, columns=values_to_fetch)
mean_df = mean_df.dropna(axis='rows')

reshaped_matrix_std = np.nanstd(results_matrix, axis=1).reshape(results_matrix.shape[0],np.prod(results_matrix.shape[2:])).T
std_df = pd.DataFrame(reshaped_matrix_std, index=name_list, columns=values_to_fetch)
std_df = std_df.dropna(axis="rows")

mean_df

In [ ]:
rounding_val = 2 if zero_one_loss else 4
rounding = f"%.{rounding_val}f"

if zero_one_loss:
    vals_to_add =['complement_error', 'test_error', 'message_len',  'best_model_complement_error','best_model_test_error',\
                       'kl_bound', 'disagreement_kl', 'full_disagreement_bound_brute_force']
else:
    vals_to_add = ['complement_huber_loss', 'test_huber_loss', 'message_len',
                       'best_model_complement_huber_loss','best_model_test_huber_loss',\
                      'huber_kl_bound', 'disagreement_softmax_huber', 'full_disagreement_softmax' ]
    
mult = 100.0 if zero_one_loss else 1.0

results_list = []
model_list = []

for i in range(mean_df.shape[0]):
    mean_vals = mean_df.iloc[i]
    std_vals = std_df.iloc[i]

    name = mean_vals.name
    first_index = name.index(" ")
    
    model_name = name[0:first_index]
    list_of_vals = []

    best_vals_mean, best_vals_std = get_best_huber_loss(model_name)
    
    for val in vals_to_add:
        if val == 'best_model_complement_huber_loss':
            temp = (rounding % round(mult*best_vals_mean['Train huber loss'],rounding_val)) + r"$\pm$"
            temp += (rounding % round(mult*best_vals_std['Train huber loss'], rounding_val))
            list_of_vals.append(temp)
            continue
        elif val == 'best_model_test_huber_loss':
            temp = (rounding % round(mult*best_vals_mean['Test huber loss'],rounding_val)) + r"$\pm$"
            temp += (rounding % round(mult*best_vals_std['Test huber loss'], rounding_val))
            list_of_vals.append(temp)
            continue
        elif val == "message_len":
            temp = (rounding % round(mean_vals['message_len'],rounding_val)) + r"$\pm$"
            temp += (rounding % round(std_vals['message_len'], rounding_val))
            list_of_vals.append(temp)
            continue
            
        temp = (rounding % round(mult*mean_vals[val],rounding_val)) + r"$\pm$"
        temp += (rounding % round(mult*std_vals[val], rounding_val))
        list_of_vals.append(temp)
    
    results_list.append(pd.Series(list_of_vals, index=vals_to_add))
    model_list.append(model_name)

if zero_one_loss:
    df = pd.DataFrame(results_list, index=model_list)
    print(df.to_latex(float_format="%.2f"))
else:
    print(pd.DataFrame(results_list, index=model_list).to_latex(float_format="%.4f"))